# 02 Preprocessing & Feature Engineering

**Purpose:**
- Clean raw data (remove anomalies, handle missing values)                     
- Create temporally-aware train/validation/test split
- Engineer basic features (temporal, popularity counts)                        
- Prepare datasets for feature engineering & modeling                          

**Outputs:**
- `train_processed.csv`: 6,046,547 rows × 21 columns (training data, oldest
events)
- `val_processed.csv`: 755,818 rows × 21 columns (validation data, middle
events)
- `test_processed.csv`: 755,819 rows × 21 columns (internal test data, newest
events)
- `test_processed.csv` (Kaggle): 19,918 rows × 21 columns (Kaggle submission
target)

**Key Preprocessing Decisions:**

1. **Data Cleaning**
    - Removed 2 rows with `media_duration = 65535` (suspicious outliers)
    - Removed 648 rows (0.009%) with invalid timestamps (ts_listen < 1B,
release_date in year 3000)
    - Final clean dataset: 7,558,184 rows

2. **Temporal Train/Val/Test Split**
    - Sorted by `ts_listen` (Unix timestamp) in ascending order
    - Split 80/10/10 by record position (ensures chronological order)
    - **Verified non-overlapping time ranges:**
      - Train:  Dec 31, 2008 → Oct 24, 2016
      - Val:    Oct 24, 2016 → Nov 23, 2016
      - Test:   Nov 23, 2016 → Nov 30, 2016
    - Class distribution stable across splits (69% → 67% → 64% listened)

3. **Feature Engineering**
    - **Temporal features:** `listen_hour`, `listen_dayofweek`, `listen_month`
(from Unix timestamp)
    - **Song age:** Days since release (handled 26K negative values in train,
filled with median=728 days)
    - **Popularity counts** (computed ONLY from train, merged to val/test):
      - `user_listen_count`: How active is this user?
      - `item_listen_count`: How popular is this song?
      - `artist_listen_count`: How well-known is this artist?
      - `genre_listen_count`: How popular is this genre?

4. **Missing Values & Cold-Start Handling**
    - **Popularity features:** Items/users not in training → NaN (filled with 0)
    - **Val set:** 13.92% cold-start rows (items/users not seen in training)
    - **Test set:** 27.71% cold-start rows (realistic for production)
    - **Song age:** 0.44% negative values in train, 0.24% in val, 0.14% in test
(replaced with median)
    - **Media duration:** 179 rows with duration=0 (replaced with median=222
sec)

**Final Feature Set (21 columns):**
- **Original:** genre_id, media_id, album_id, context_type, platform_name,
platform_family, media_duration, listen_type, user_gender, user_id, artist_id,
user_age, is_listened
- **Engineered:** listen_hour, listen_dayofweek, listen_month, song_age_days
- **Popularity:** user_listen_count, item_listen_count, artist_listen_count,
genre_listen_count

**Critical Design Choices:**
- Features computed ONLY from train set (prevents data leakage)
- Cold-start handled explicitly (0 for unseen items, documented %)
- Temporal split matches real-world scenario (train on past, test on future)
- All transformations applied consistently to train/val/test

**Next:** 03_baseline_model.ipynb (establish performance baselines)

In [1]:
from google.colab import drive
#drive.mount('/content/gdrive')
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



#file_path = '/content/gdrive/MyDrive/Recommender Systems (Project)/train.csv'


file_path = '/content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/train.csv'
df_train = pd.read_csv(file_path)

## Preprocessing

### Verify data loaded

In [3]:
df_train.shape

(7558834, 15)

In [4]:
df_train.head()

,genre_id,ts_listen,media_id,album_id,context_type,release_date,platform_name,platform_family,media_duration,listen_type,user_gender,user_id,artist_id,user_age,is_listened
0,25471,1480597215,222606,41774,12,20040704,1,0,223,0,0,9241,55164,29,0
1,25571,1480544735,250467,43941,0,20060301,2,1,171,0,0,16547,55830,30,1
2,16,1479563953,305197,48078,1,20140714,2,1,149,1,1,7665,2704,29,1
3,7,1480152098,900502,71521,0,20001030,0,0,240,0,1,1580,938,30,0
4,7,1478368974,542335,71718,0,20080215,0,0,150,0,1,1812,2939,24,1


### Drop the 2 outliers

In [5]:
# Remove duration outliers
df_train = df_train[df_train['media_duration'] != 65535]
print(f"Shape after removing outliers: {df_train.shape}")

Shape after removing outliers: (7558832, 15)


### Check for duration = 0

In [6]:
zero_duration_count = (df_train['media_duration'] == 0).sum()
print(f"Rows with media_duration = 0: {zero_duration_count}")

# Look at a few examples
if zero_duration_count > 0:
    print(df_train[df_train['media_duration'] == 0].head())

Rows with media_duration = 0: 179
        genre_id   ts_listen   media_id  album_id  context_type  release_date  \
119081    215245  1477979474   85833267   8656389             0      20120101   
119082    215245  1478764819   85833267   8656389             0      20120101   
119083    215245  1479233500   85833267   8656389             0      20120101   
119084    215245  1478898436   85833267   8656389             0      20120101   
566515         0  1478266454  103346690  10743612             0      20150101   

        platform_name  platform_family  media_duration  listen_type  \
119081              1                0               0            0   
119082              1                0               0            0   
119083              1                0               0            0   
119084              1                0               0            0   
566515              0                0               0            0   

        user_gender  user_id  artist_id  user_age  i

### Check timestamps

In [7]:
print(f"ts_listen range: {df_train['ts_listen'].min()} to {df_train['ts_listen'].max()}")
print(f"release_date range: {df_train['release_date'].min()} to {df_train['release_date'].max()}")

ts_listen range: 1 to 1480635845
release_date range: 19000101 to 30000101


  ts_listen (**1 to 1480635845**):                                                                    
  - Unix timestamp format                                                                         
  - 1480635845 ≈ November 2016                                                                    
  - Starts at 1 — suspicious (that's not a valid listening time, likely errors/placeholders)      
  - These are real listening sessions from 2016                                                   
                                                                                                  
  release_date (**19000101 to 30000101**):                                                            
  - YYYYMMDD format                                                                               
  - Real dates: ~1900-2020 (reasonable for music history)                                         
  - **But 30000101**? That's year 3000 — clearly a placeholder for unknown release dates!

### How many suspicious values?

In [8]:
# Check ts_listen = 0 or = 1
print(f"ts_listen < 1000000000: {(df_train['ts_listen'] < 1000000000).sum()}")

# Check release_date placeholders (year 3000 or higher)
print(f"release_date >= 30000101: {(df_train['release_date'] >= 30000101).sum()}")

# Check release_date before 1900
print(f"release_date < 19000101: {(df_train['release_date'] < 19000101).sum()}")

ts_listen < 1000000000: 631
release_date >= 30000101: 17
release_date < 19000101: 0


  `ts_listen < 1000000000` (**631 rows**):                                                              
  - Only 0.008% of data                                                                           
  - These are errors or system placeholders                                                       
  - Too small to be valid Unix timestamps                                                         
                                                                                                  
  `release_date >= 30000101` (**17 rows**):                                                             
  - Only 0.0002% of data                                                                          
  - Clear placeholders for unknown release dates                                                  
                                                                                                  
  `release_date < 19000101`: None — good!

In [9]:
# Identify problematic rows
problematic_ts = df_train['ts_listen'] < 1000000000
problematic_date = df_train['release_date'] >= 30000101

# Check if any row has BOTH problems
both = problematic_ts & problematic_date
either = problematic_ts | problematic_date

print(f"Rows with problematic ts_listen: {problematic_ts.sum()}")
print(f"Rows with problematic release_date: {problematic_date.sum()}")
print(f"Rows with BOTH problems: {both.sum()}")
print(f"Rows with EITHER problem: {either.sum()}")

Rows with problematic ts_listen: 631
Rows with problematic release_date: 17
Rows with BOTH problems: 0
Rows with EITHER problem: 648


Since 648 rows total (0.009% of data), the simplest approach is to drop them.

In [10]:
# Drop rows with problematic values
df_train = df_train[~either]

# Verify
print(f"Shape after cleaning: {df_train.shape}")
print(f"Rows removed: {7558832 - len(df_train)}")

Shape after cleaning: (7558184, 15)
Rows removed: 648


In [11]:
# Quick verification
print(f"\nts_listen range: {df_train['ts_listen'].min()} to {df_train['ts_listen'].max()}")
print(f"release_date range: {df_train['release_date'].min()} to {df_train['release_date'].max()}")
print(f"media_duration range: {df_train['media_duration'].min()} to {df_train['media_duration'].max()}")
print(f"\nNo missing values: {df_train.isnull().sum().sum() == 0}")


ts_listen range: 1230764491 to 1480635845
release_date range: 19000101 to 20170313
media_duration range: 0 to 9158

No missing values: True


## Split Data into train/Val/Test

In [12]:
# split data 80/10/10 chronologically using ts_listen by sorting and taking the oldest to newest.

df_sorted = df_train.sort_values("ts_listen").reset_index(drop=True)

n = len(df_sorted)
train_cut = int(0.8 * n)
val_cut   = int(0.9 * n)

train_data = df_sorted.iloc[:train_cut].copy()
val_data   = df_sorted.iloc[train_cut:val_cut].copy()
test_data  = df_sorted.iloc[val_cut:].copy()

print("Shapes:", train_data.shape, val_data.shape, test_data.shape)
print("ts_listen ranges:")
print(" train:", train_data["ts_listen"].min(), train_data["ts_listen"].max())
print("   val:", val_data["ts_listen"].min(), val_data["ts_listen"].max())
print("  test:", test_data["ts_listen"].min(), test_data["ts_listen"].max())

print("Target means:")
print(" train:", train_data["is_listened"].mean())
print("   val:", val_data["is_listened"].mean())
print("  test:", test_data["is_listened"].mean())

Shapes: (6046547, 15) (755818, 15) (755819, 15)
ts_listen ranges:
 train: 1230764491 1479895930
   val: 1479895930 1480189309
  test: 1480189309 1480635845
Target means:
 train: 0.6906505481558317
   val: 0.6729675662659529
  test: 0.6419777751022401


## Feature Engineering

### temporal features from ts_listen

In [13]:
# Convert Unix timestamp to datetime

def add_time_features(df):
    df = df.copy()
    df["listen_datetime"] = pd.to_datetime(df["ts_listen"], unit="s")
    df["listen_hour"] = df["listen_datetime"].dt.hour
    df["listen_dayofweek"] = df["listen_datetime"].dt.dayofweek  # 0=Mon, 6=Sun
    df["listen_month"] = df["listen_datetime"].dt.month
    return df

train_data = add_time_features(train_data)
val_data   = add_time_features(val_data)
test_data  = add_time_features(test_data)

print(train_data[["ts_listen","listen_datetime","listen_hour","listen_dayofweek","listen_month"]].head())


    ts_listen     listen_datetime  listen_hour  listen_dayofweek  listen_month
0  1230764491 2008-12-31 23:01:31           23                 2            12
1  1230764553 2008-12-31 23:02:33           23                 2            12
2  1230764583 2008-12-31 23:03:03           23                 2            12
3  1230764600 2008-12-31 23:03:20           23                 2            12
4  1230764709 2008-12-31 23:05:09           23                 2            12


In [17]:
# verify that there is no overlap and the split is chronological
col= "listen_datetime"

tmin, tmax = train_data[col].min(), train_data[col].max()
vmin, vmax = val_data[col].min(), val_data[col].max()
smin, smax = test_data[col].min(), test_data[col].max()

print("Ranges:")
print(" train:", tmin, "->", tmax)
print("   val:", vmin, "->", vmax)
print("  test:", smin, "->", smax)

# boundary checks
print("\nBoundary checks (should be True):")
print(" train_max <= val_min :", tmax <= vmin)
print(" val_max   <= test_min:", vmax <= smin)

Ranges:
 train: 2008-12-31 23:01:31 -> 2016-11-23 10:12:10
   val: 2016-11-23 10:12:10 -> 2016-11-26 19:41:49
  test: 2016-11-26 19:41:49 -> 2016-12-01 23:44:05

Boundary checks (should be True):
 train_max <= val_min : True
 val_max   <= test_min: True


In [ ]:
# Convert release_date to datetime
for df in (train_data, val_data, test_data):
    df["release_datetime"] = pd.to_datetime(
        df["release_date"].astype(str),
        format="%Y%m%d",
        errors="coerce" )

# Calculate song age (days since release at time of listening)
    df["song_age_days"] = (df["listen_datetime"] - df["release_datetime"]).dt.days


# Check results
print(f"Train song_age range: {train_data['song_age_days'].min()} to {train_data['song_age_days'].max()} days")
print(train_data[["release_date", "release_datetime", "listen_datetime", "song_age_days"]].head())

print(f"Test song_age range: {test_data['song_age_days'].min()} to {test_data['song_age_days'].max()} days")
print(test_data[["release_date", "release_datetime", "listen_datetime", "song_age_days"]].head())

print(f"Val song_age range: {val_data['song_age_days'].min()} to {val_data['song_age_days'].max()} days")
print(val_data[["release_date", "release_datetime", "listen_datetime", "song_age_days"]].head())

Train song_age range: -2857 to 42695 days
   release_date release_datetime     listen_datetime  song_age_days
0      20160916       2016-09-16 2008-12-31 23:01:31          -2816
1      20160624       2016-06-24 2008-12-31 23:02:33          -2732
2      20050517       2005-05-17 2008-12-31 23:03:03           1324
3      20160930       2016-09-30 2008-12-31 23:03:20          -2830
4      20031231       2003-12-31 2008-12-31 23:05:09           1827
Test song_age range: -102 to 42703 days
         release_date release_datetime     listen_datetime  song_age_days
6802365      20161021       2016-10-21 2016-11-26 19:41:49             36
6802366      19801231       1980-12-31 2016-11-26 19:41:49          13114
6802367      20151030       2015-10-30 2016-11-26 19:41:50            393
6802368      20041018       2004-10-18 2016-11-26 19:41:50           4422
6802369      20051219       2005-12-19 2016-11-26 19:41:50           3995
Val song_age range: -102 to 42698 days
         release_date relea

In some cases the release date appears after the listen data (negative values/meta data inconsistencies).

In [ ]:
# How many negative song_age values?
for name, df in [("train", train_data), ("val", val_data), ("test", test_data)]:
    n = len(df)
    n_nan = df["song_age_days"].isna().sum()
    n_neg = (df["song_age_days"] < 0).sum()

    print(f"\n{name.upper()}:")
    print(f"  rows: {n:,}")
    print(f"  NaN song_age_days: {n_nan:,} ({100*n_nan/n:.4f}%)")
    print(f"  negative song_age_days: {n_neg:,} ({100*n_neg/n:.4f}%)")
    print(f"  min/max (ignoring NaN): {df['song_age_days'].min()} / {df['song_age_days'].max()}")


TRAIN:
  rows: 6,046,547
  NaN song_age_days: 0 (0.0000%)
  negative song_age_days: 26,303 (0.4350%)
  min/max (ignoring NaN): -2857 / 42695

VAL:
  rows: 755,818
  NaN song_age_days: 0 (0.0000%)
  negative song_age_days: 1,825 (0.2415%)
  min/max (ignoring NaN): -102 / 42698

TEST:
  rows: 755,819
  NaN song_age_days: 0 (0.0000%)
  negative song_age_days: 1,022 (0.1352%)
  min/max (ignoring NaN): -102 / 42703


We observe small percentage of negative values across all splits (train: 0.435%, val:0.2415%, test:0.1352%). As mentioned above negative values indicate inconsistent metadata where relase date is after listen date. In the next section we compute the media song_age_days from the training set using positive values and replace the negatice values in the splits.

In [ ]:
# Replace negative song_age with median (only for positive values)
for df in (train_data, val_data, test_data):
    df["song_age_days"] = pd.to_numeric(df["song_age_days"], errors="coerce")

# train-only median from non-negative values
median_song_age = train_data.loc[train_data["song_age_days"] >= 0, "song_age_days"].median()
print("Median song_age (non-negative only, train):", median_song_age)

# replace negatives + fill NaNs in all splits
for df in (train_data, val_data, test_data):
    df.loc[df["song_age_days"] < 0, "song_age_days"] = median_song_age
    df["song_age_days"] = df["song_age_days"].fillna(median_song_age)

# verify
for name, df in [("TRAIN", train_data), ("VAL", val_data), ("TEST", test_data)]:
    print(f"\n{name}:")
    print("  negatives:", (df["song_age_days"] < 0).sum())
    print("  NaNs:", df["song_age_days"].isna().sum())
    print("  min/max:", df["song_age_days"].min(), df["song_age_days"].max())

Median song_age (non-negative only, train): 728.0

TRAIN:
  negatives: 0
  NaNs: 0
  min/max: 0 42695

VAL:
  negatives: 0
  NaNs: 0
  min/max: 0 42698

TEST:
  negatives: 0
  NaNs: 0
  min/max: 0 42703


### user/item popularity features

In [ ]:
# 1. USER ACTIVITY: How many songs has each user listened to?
user_listen_count = train_data.groupby('user_id').size().reset_index(name='user_listen_count')

train_data = train_data.merge(user_listen_count, on="user_id", how="left")
val_data   = val_data.merge(user_listen_count, on="user_id", how="left")
test_data  = test_data.merge(user_listen_count, on="user_id", how="left")

In [ ]:
# 2. ITEM POPULARITY: How many users have listened to each media_id?

item_listen_count = train_data.groupby("media_id").size().reset_index(name="item_listen_count")

train_data = train_data.merge(item_listen_count, on="media_id", how="left")
val_data   = val_data.merge(item_listen_count, on="media_id", how="left")
test_data  = test_data.merge(item_listen_count, on="media_id", how="left")


In [ ]:
# 3. ARTIST POPULARITY: How many times has each artist been listened to?
artist_listen_count = train_data.groupby("artist_id").size().reset_index(name="artist_listen_count")

train_data = train_data.merge(artist_listen_count, on="artist_id", how="left")
val_data   = val_data.merge(artist_listen_count, on="artist_id", how="left")
test_data  = test_data.merge(artist_listen_count, on="artist_id", how="left")

In [ ]:
# 4. GENRE POPULARITY: How many times has each genre been listened to?
genre_listen_count = train_data.groupby("genre_id").size().reset_index(name="genre_listen_count")

train_data = train_data.merge(genre_listen_count, on="genre_id", how="left")
val_data   = val_data.merge(genre_listen_count, on="genre_id", how="left")
test_data  = test_data.merge(genre_listen_count, on="genre_id", how="left")

In [ ]:
print([c for c in train_data.columns if "listen_count" in c or "count" in c])

['user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']


In [ ]:
print("Popularity features created (TRAIN):")
print(train_data[['user_id','user_listen_count','media_id','item_listen_count',
                  'artist_id','artist_listen_count','genre_id','genre_listen_count']].head(10))
print("Train shape:", train_data.shape)

print("\nPopularity features created (VAL):")
print(val_data[['user_id','user_listen_count','media_id','item_listen_count',
                'artist_id','artist_listen_count','genre_id','genre_listen_count']].head(10))
print("Val shape:", val_data.shape)

print("\nPopularity features created (TEST):")
print(test_data[['user_id','user_listen_count','media_id','item_listen_count',
                 'artist_id','artist_listen_count','genre_id','genre_listen_count']].head(10))
print("Test shape:", test_data.shape)

Popularity features created (TRAIN):
   user_id  user_listen_count   media_id  item_listen_count  artist_id  \
0    19655                  2  132198352                350         17   
1    11748                140  125599920               1238     523462   
2     3670                506     670673                  1      64431   
3       42               2631  133120054                219    1623794   
4     3670                506    1831862                  1    1319539   
5    19655                  2  132198354                 35         17   
6    19676                  1  132123630              11517    1519461   
7       42               2631    5639256               2749      75001   
8     3670                506    1642314                  1     316709   
9     3670                506    3283973                  1     214795   

   artist_listen_count  genre_id  genre_listen_count  
0                 7502         0             2967220  
1                 3957         0      

In [ ]:
# the features are created only using train data to avoid leakage
pop_cols = ["user_listen_count","item_listen_count","artist_listen_count","genre_listen_count"]
for name, df in [("VAL", val_data), ("TEST", test_data)]:
    print(name, {c: int(df[c].isna().sum()) for c in pop_cols if c in df.columns})

VAL {'user_listen_count': 66030, 'item_listen_count': 46331, 'artist_listen_count': 5779, 'genre_listen_count': 131}
TEST {'user_listen_count': 168874, 'item_listen_count': 56262, 'artist_listen_count': 6769, 'genre_listen_count': 123}


In [ ]:
pop_cols = ["user_listen_count", "item_listen_count", "artist_listen_count", "genre_listen_count"]

for df in (train_data, val_data, test_data):
    for c in pop_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int32")

# Verify NaNs are gone + show % of cold-start rows
for name, df in [("TRAIN", train_data), ("VAL", val_data), ("TEST", test_data)]:
    nan_counts = {c: int(df[c].isna().sum()) for c in pop_cols}
    cold_rows = int((df[pop_cols] == 0).any(axis=1).sum())  # rows where at least one pop feature is 0
    print(f"{name}: NaNs {nan_counts} | rows with any 0-pop feature: {cold_rows} ({100*cold_rows/len(df):.2f}%)")

TRAIN: NaNs {'user_listen_count': 0, 'item_listen_count': 0, 'artist_listen_count': 0, 'genre_listen_count': 0} | rows with any 0-pop feature: 0 (0.00%)
VAL: NaNs {'user_listen_count': 0, 'item_listen_count': 0, 'artist_listen_count': 0, 'genre_listen_count': 0} | rows with any 0-pop feature: 105185 (13.92%)
TEST: NaNs {'user_listen_count': 0, 'item_listen_count': 0, 'artist_listen_count': 0, 'genre_listen_count': 0} | rows with any 0-pop feature: 209455 (27.71%)


### Summary
 After spliting the data in train, validation and test data, we computed interactions from the train slit (user_listen_count, item_listen_count, artist_listen_count,genre_listen_count). Enteties not observed in training prooduced missing values ( no observed history). These values are imputed with 0, meaning no observed history in training.

 After imputation val_data and test_data contain 13.92% and 27.71% entries with no observed history in train_data.

### Final Cleanup

In [ ]:
# Replace media_duration == 0 using train median (non-zero)
median_duration = train_data.loc[train_data["media_duration"] > 0, "media_duration"].median()
print("Median duration (non-zero, train):", median_duration)

for name, df in [("TRAIN", train_data), ("VAL", val_data), ("TEST", test_data)]:
    df.loc[df["media_duration"] == 0, "media_duration"] = median_duration
    print(name, "zeros after:", int((df["media_duration"] == 0).sum()))

Median duration (non-zero, train): 222.0
TRAIN zeros after: 0
VAL zeros after: 0
TEST zeros after: 0


In [ ]:
# Drop original timestamp columns (we extracted features from them)
columns_to_drop = ["ts_listen", "listen_datetime", "release_date", "release_datetime"]

for name, df in [("TRAIN", train_data), ("VAL", val_data), ("TEST", test_data)]:
    drop_cols = [c for c in columns_to_drop if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)

    print(f"\n{name} final shape: {df.shape}")
    print(f"{name} columns ({len(df.columns)}):")
    print(df.columns.tolist())


TRAIN final shape: (6046547, 21)
TRAIN columns (21):
['genre_id', 'media_id', 'album_id', 'context_type', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age', 'is_listened', 'listen_hour', 'listen_dayofweek', 'listen_month', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']

VAL final shape: (755818, 21)
VAL columns (21):
['genre_id', 'media_id', 'album_id', 'context_type', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age', 'is_listened', 'listen_hour', 'listen_dayofweek', 'listen_month', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']

TEST final shape: (755819, 21)
TEST columns (21):
['genre_id', 'media_id', 'album_id', 'context_type', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id',

## Modifying Test-Dataset

In [ ]:
# LOAD TEST DATA
# test_path = '/content/gdrive/MyDrive/Recommender Systems (Project)/test.csv'
test_path = '/content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/test.csv'
df_test = pd.read_csv(test_path)

print(f"Test dataset shape: {df_test.shape}")
print(f"Test columns: {df_test.columns.tolist()}")
print(f"Does test have 'is_listened'? {'is_listened' in df_test.columns}")

Test dataset shape: (19918, 15)
Test columns: ['sample_id', 'genre_id', 'ts_listen', 'media_id', 'album_id', 'context_type', 'release_date', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age']
Does test have 'is_listened'? False


In [ ]:
# Apply same transformations to test data
df_test_processed = df_test.copy()

# 1. Drop problematic rows (same thresholds as train)
df_test_processed = df_test_processed[df_test_processed['ts_listen'] >= 1000000000]
df_test_processed = df_test_processed[df_test_processed['release_date'] < 30000101]

# 2. Convert timestamps
df_test_processed['listen_datetime'] = pd.to_datetime(df_test_processed['ts_listen'], unit='s')
df_test_processed['listen_hour'] = df_test_processed['listen_datetime'].dt.hour
df_test_processed['listen_dayofweek'] = df_test_processed['listen_datetime'].dt.dayofweek
df_test_processed['listen_month'] = df_test_processed['listen_datetime'].dt.month

# 3. Calculate song age
df_test_processed['release_datetime'] = pd.to_datetime(df_test_processed['release_date'].astype(str), format='%Y%m%d')
df_test_processed['song_age_days'] = (df_test_processed['listen_datetime'] - df_test_processed['release_datetime']).dt.days

# Replace negative song_age
df_test_processed.loc[df_test_processed['song_age_days'] < 0, 'song_age_days'] = 755.0

# 4. Replace duration = 0
df_test_processed.loc[df_test_processed['media_duration'] == 0, 'media_duration'] = 222.0

# 5. Add popularity features
df_test_processed = df_test_processed.merge(user_listen_count, on='user_id', how='left')
df_test_processed = df_test_processed.merge(item_listen_count, on='media_id', how='left')
df_test_processed = df_test_processed.merge(artist_listen_count, on='artist_id', how='left')
df_test_processed = df_test_processed.merge(genre_listen_count, on='genre_id', how='left')

# 6. Drop original timestamp columns
df_test_processed = df_test_processed.drop(columns=['ts_listen', 'listen_datetime', 'release_date', 'release_datetime'])

print(f"Test processed shape: {df_test_processed.shape}")
print(f"Test columns: {df_test_processed.columns.tolist()}")

Test processed shape: (19918, 21)
Test columns: ['sample_id', 'genre_id', 'media_id', 'album_id', 'context_type', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age', 'listen_hour', 'listen_dayofweek', 'listen_month', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']


### Save Datasets

In [ ]:
# Save processed train data
# train_output_path = '/content/gdrive/MyDrive/Recommender Systems (Project)/train_processed.csv'

import os
output_path ='/content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)'

os.makedirs(output_path, exist_ok=True)

train_processed = os.path.join(output_path, "train_processed.csv")

train_data = os.path.join(output_path, "train_data.csv")

test_data = os.path.join(output_path, "test_data.csv")

val_data = os.path.join(output_path, "val_data.csv")

test_processed = os.path.join(output_path, "test_processed.csv")

# Save

df_train.to_csv(train_output_path, index=False)
df_train

Train saved: /content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/train_processed.csv
Train Split saved: /content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/train_processed.csv
Val Split saved: /content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/train_processed.csv
Test Split saved: /content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/train_processed.csv


OSError: Cannot save file into a non-existent directory: '/content/gdrive/MyDrive/Recommender Systems (Project)'